# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`
This notebook provides a guided template for loading and exploring the FAIR² Mental Health Survey Dataset from Kilifi County, Kenya using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL (JSON-LD):

`https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`. This section sets up the dataset object and prints an overview.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata.to_json()
print("--- Dataset Info ---")
print("Name:", metadata.get('name'))
print("Description:", metadata.get('description'))
print("Identifier:", metadata.get('identifier'))
print("Version:", metadata.get('version'))
print("Published:", metadata.get('datePublished'))
print("Keywords:", metadata.get('keywords'))

## 2. Data Overview
Review available record sets, fields, and their `@id`s.

This section lists the record sets, fields, and columns, referencing all entities by their `@id` fields, as per the Croissant schema.

In [ ]:
# Explore record sets and fields using their @id values
# List all record sets; each usually has columns or fields
record_sets = []

# In some Croissant schemas, dataset.metadata has a 'recordSet' entry listing all record sets
if 'recordSet' in metadata and isinstance(metadata['recordSet'], list):
    for rs in metadata['recordSet']:
        record_sets.append(rs['@id'])
else:
    # If no explicit recordSet listed, we'll attempt to discover record sets via mlcroissant
    record_sets = dataset.record_sets

print("Record sets @id(s):")
for rs_id in record_sets:
    print(f"- {rs_id}")

# List fields/columns for each record set
for rs_id in record_sets:
    info = dataset.record_set_info(rs_id)
    print(f"\nFields/Columns for record set @id={rs_id}:")
    for column in info['columns']:
        print(f"  - {column['@id']} ({column.get('name','')}) : {column.get('dataType','')} - source: {column.get('source','')}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview above.
Here, we'll select the first record set (if available) and extract its content.

In [ ]:
# For demonstration, select the first record set and extract records
dataframes = {}

if record_sets:
    selected_record_set_id = record_sets[0]
    print(f"\nExtracting records for record set @id={selected_record_set_id}")
    records = list(dataset.records(record_set=selected_record_set_id))
    df = pd.DataFrame(records)
    dataframes[selected_record_set_id] = df
    print("Columns:", df.columns.tolist())
    print(df.head())
else:
    print("No record sets found in the schema.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

We will:
- Filter records based on a numeric field (e.g., GAD-7 score)
- Normalize the numeric field
- Group records by a categorical field (e.g., level_of_education)

In [ ]:
# Choose numeric and group fields by their @id
# Find likely IDs for GAD-7 score and level_of_education

# List all columns for reference
if record_sets:
    cols = dataframes[selected_record_set_id].columns.tolist()
    print("Available columns (by field @id):", cols)
    # Replace these with the actual @id from the overview if you want
    numeric_field_id = None
    group_field_id = None
    # Attempt to auto-select
    for col in cols:
        if 'gad_7_score' in col.lower():
            numeric_field_id = col
        if 'level_of_education' in col.lower():
            group_field_id = col
    # Fallback: choose the first numeric and group field if not found
    if numeric_field_id is None:
        # Try to pick a numeric column (int/float type)
        for col in cols:
            if pd.api.types.is_numeric_dtype(dataframes[selected_record_set_id][col]):
                numeric_field_id = col
                break
    if group_field_id is None:
        for col in cols:
            if 'education' in col.lower() or 'level_of_education' in col.lower():
                group_field_id = col

    print("Numeric field @id:", numeric_field_id)
    print("Group field @id:", group_field_id)

    # Filtering
    threshold = 10
    if numeric_field_id:
        filtered_df = dataframes[selected_record_set_id][dataframes[selected_record_set_id][numeric_field_id] > threshold]
        print(f"Filtered records with {numeric_field_id} > {threshold}:")
        print(filtered_df.head())

        # Normalization
        filtered_df[numeric_field_id + '_normalized'] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"Normalized {numeric_field_id} for filtered records:")
        print(filtered_df[[numeric_field_id, numeric_field_id + '_normalized']].head())

        # Group by
        if group_field_id and group_field_id in filtered_df.columns:
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
            print(f"Grouped data by {group_field_id} (mean {numeric_field_id}):")
            print(grouped_df.head())
else:
    print("No data loaded. Please check available record sets.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset, e.g., distribution of GAD-7 scores or PHQ-9 scores, grouped by education level.


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Visualization example: Distribution of GAD-7 scores
if record_sets and numeric_field_id:
    plt.figure(figsize=(8,4))
    sns.histplot(dataframes[selected_record_set_id][numeric_field_id].dropna(), kde=True, bins=15, color='skyblue', edgecolor='black')
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

    # Boxplot of GAD-7 scores by education level (if available)
    if group_field_id:
        plt.figure(figsize=(10,5))
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=dataframes[selected_record_set_id])
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
In this notebook, we used the `mlcroissant` library to load, explore, and process the FAIR² Mental Health Survey Dataset from Kilifi County, Kenya.

- We reviewed available entities by their `@id` and extracted survey records for analysis.
- Applied basic EDA: filtering, normalization, and grouping (demonstrated using GAD-7 scores and education levels).
- Visualized numeric distributions and compared across groups.

This workflow demonstrates how Croissant schemas enable machine-actionable, reproducible exploration and analysis of complex datasets.

For deeper analysis, you can extend EDA to include other fields such as PHQ-9, PSQ, demographics, or trend analyses across survey dates.